# Lecture 3.5 — Tool Context: Accessing Run State Inside a Tool

**Section 03 — Tools: Extending Agent Capabilities**

In this notebook you will learn how the OpenAI Agents SDK lets you pass application-level state into your tool functions at runtime using `RunContextWrapper` and `ToolContext`. By the end you will be able to:

- Define a typed context dataclass and pass it to `Runner.run()` via the `context=` argument
- Declare `RunContextWrapper[T]` as a tool's first parameter so the SDK injects context automatically and excludes it from the JSON schema
- Understand why the context parameter is invisible to the model — it never appears in the tool schema the LLM sees
- Use `ToolContext[T]` when you need tool-specific metadata such as `tool_name`, `tool_call_id`, and `tool_arguments`
- Mutate the context object inside a tool to accumulate state across multiple tool calls in a single run
- Understand the thread-safety constraint: each concurrent run needs its own context instance

## Cell 1 — Install the SDK

📌 **Notebook update notice:** this lecture's markdown references `openai-agents==0.17.7` as the pinned version. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version stated above. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one mentioned in the recording.

This cell installs `openai-agents`, the Python SDK for the OpenAI Agents framework. The version is pinned so that every example in this notebook runs exactly as written, regardless of what future releases may change.

**Why pin the version?** Reproducibility. Course examples are tested against a specific release. Pinning ensures you see the same behaviour shown in the lecture recording.

**Want to use a different version?** Remove the `==0.18.3` pin to install the latest release, or replace it with the version you prefer

If the package is already present in your current Colab session at this version, the install cell completes instantly and moves on.

In [ ]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

## Cell 2 — API Key Setup

The SDK reads your OpenAI API key from the `OPENAI_API_KEY` environment variable. This cell retrieves the key from Google Colab Secrets and writes it to the environment.

**How to add your key in Colab:**
1. Click the **key icon** (🔑) in the left sidebar to open the Secrets panel.
2. Click **Add new secret**.
3. Set the **Name** to `OPENAI_API_KEY`.
4. Paste your key as the **Value**.
5. Toggle **Notebook access** to ON for this notebook.
6. Run this cell.

**Running locally?** Set the variable in your terminal before launching Jupyter:
```bash
export OPENAI_API_KEY="sk-..."
```

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Model Name

Every `Agent` in this notebook uses `MODEL_NAME` instead of a hardcoded string. Change the value here and every agent picks it up automatically — no need to hunt through the notebook.

`gpt-5.4-mini` is the SDK default for low-latency agent workflows. It defaults to `reasoning.effort="none"` and `verbosity="low"`, which keeps costs and latency low for demos and course exercises.

See the full list of available models at: https://platform.openai.com/docs/models

In [ ]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

| Import | Source | Purpose |
|--------|--------|---------|
| `json` | standard library | Pretty-print JSON schemas |
| `dataclass` | standard library | Define lightweight typed context objects |
| `Reasoning` | `openai.types.shared` | Configure reasoning effort on GPT-5 models |
| `Agent` | `agents` | Core agent class |
| `ModelSettings` | `agents` | Attach model configuration to an agent |
| `RunContextWrapper` | `agents` | Type annotation for context-receiving tool parameters |
| `Runner` | `agents` | Executes agent runs |
| `function_tool` | `agents` | Decorator that turns a Python function into an SDK tool |
| `ToolContext` | `agents.tool_context` | Subclass of `RunContextWrapper` that adds tool-specific metadata |

**Critical import note:** `ToolContext` lives in `agents.tool_context`, not in the top-level `agents` package. Importing it from `agents` directly raises an `ImportError`. Always use `from agents.tool_context import ToolContext`.

In [ ]:
import json
from dataclasses import dataclass

from openai.types.shared import Reasoning

from agents import (
    Agent,
    ModelSettings,
    RunContextWrapper,
    Runner,
    function_tool,
)
from agents.tool_context import ToolContext

## Cell 5 — How Context Works in the SDK

Before writing any tools, it is worth understanding exactly what happens at the SDK level.

### The context flow

1. You create any Python object — a dataclass, a Pydantic model, a plain class.
2. You pass it to `Runner.run(agent, input, context=your_object)`.
3. The SDK wraps it in a `RunContextWrapper[T]` and routes that wrapper to every tool function, lifecycle hook, and guardrail in the run.

### What `RunContextWrapper` exposes

| Attribute | Type | What it gives you |
|-----------|------|--------------------|
| `wrapper.context` | `T` | Your app-defined object — the one you passed to `Runner.run()` |
| `wrapper.usage` | `Usage` | Aggregated token and request counts for this run |
| `wrapper.tool_input` | `Any` | Structured input when this run is executing inside `Agent.as_tool()` |

### How the SDK detects the context parameter

When `@function_tool` processes your function, it inspects the first parameter's type annotation.
If the annotation is `RunContextWrapper` or `ToolContext` (or a subclass), the SDK:
- sets `takes_context = True` on the function schema
- **excludes that parameter from the JSON schema entirely**
- injects the wrapper automatically when invoking the tool

The model never sees the context parameter. It cannot pass a value for it. The SDK handles it entirely.

### Rules

- The context parameter is **only valid as the first parameter**. Placing `RunContextWrapper` or `ToolContext` anywhere else raises a `UserError` at decoration time.
- All tools, hooks, and guardrails in a single run must use the **same context type** `T`.
- All tools share the **same context instance** — mutations made in one tool call are visible to subsequent calls in the same run.
- Context is **not sent to the LLM** — it is purely local Python state.
- Context is **not thread-safe** — each concurrent run must have its own context instance.

### `RunContextWrapper` vs `ToolContext`

| Use | When |
|-----|------|
| `RunContextWrapper[T]` | You only need `ctx.context` (your app state) and `ctx.usage` |
| `ToolContext[T]` | You also need `tool_name`, `tool_call_id`, `tool_arguments` for auditing, logging, or idempotency |

`ToolContext` is a subclass of `RunContextWrapper`. It adds these fields for every tool call:

| Field | Type | Description |
|-------|------|-------------|
| `tool_name` | `str` | Name of the tool being invoked |
| `tool_call_id` | `str` | Unique ID for this specific call — useful for idempotency |
| `tool_arguments` | `str` | Raw JSON arguments string |
| `tool_call` | `ResponseFunctionToolCall \| None` | Full tool call object |
| `tool_namespace` | `str \| None` | Responses API namespace |
| `agent` | `AgentBase \| None` | The agent invoking this tool |
| `run_config` | `RunConfig \| None` | The active run configuration |

## Cell 6 — Basic Context Access with `RunContextWrapper`

This is the foundational pattern. We define a `UserContext` dataclass to hold per-user data, then write a tool whose first parameter is `RunContextWrapper[UserContext]`. The SDK detects this, excludes it from the schema, and injects it automatically.

Key observations:
- `get_personalised_greeting` has **no JSON-visible parameters** — the model calls it with an empty arguments object
- `ctx.context` gives us the `UserContext` instance we passed to `Runner.run()`
- Context is registered once, at `Runner.run(..., context=ctx)` — not on the agent or the tool

In [ ]:
@dataclass
class UserContext:
    # TODO: add user_id, username, subscription_tier fields
    pass

@function_tool
def get_personalised_greeting(
    ctx: RunContextWrapper[UserContext],
) -> str:
    """Returns a personalised greeting for the current user."""
    # TODO: read ctx.context and return a greeting string using username and subscription_tier
    pass


greeting_agent = Agent(
    name="Greeting Agent",
    instructions=(
        "You are a helpful assistant. "
        "Greet the user personally using the greeting tool."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    # TODO: pass get_personalised_greeting as the tools list
)

# TODO: instantiate UserContext for Priya, user_id u001, subscription_tier Premium

result = await Runner.run(
    greeting_agent,
    "Please greet me.",
    # TODO: pass user_ctx as the context argument
)
print(result.final_output)

## Cell 7 — Inspect the Schema: Context Excluded

This cell makes the exclusion concrete. We print the JSON schema the SDK generated for `get_personalised_greeting`. The `properties` object will be empty — the model sees a tool with no arguments at all.

This is by design. The SDK detected `RunContextWrapper` as the first parameter, set `takes_context=True`, and stripped it from the schema before the model ever saw it. The injection happens entirely on the Python side, invisible to the LLM.

In [ ]:
print("JSON schema for get_personalised_greeting:")
print(json.dumps(get_personalised_greeting.params_json_schema, indent=2))

## Cell 8 — Context with Additional Parameters

A tool can accept a context parameter **and** regular JSON-visible parameters at the same time. The context must be first — the remaining parameters appear in the schema normally and are filled in by the model.

In `search_user_data`:
- `ctx` is first, typed as `RunContextWrapper[UserContext]` — excluded from schema, injected automatically
- `query` (required) and `max_results` (optional, default 5) appear in the schema — the model provides them

We print the schema to confirm: `ctx` is absent, `query` and `max_results` are present.

In [ ]:
@function_tool
def search_user_data(
    ctx: RunContextWrapper[UserContext],
    query: str,
    max_results: int = 5,
) -> str:
    """Searches the user's personal data for a given query.

    Args:
        query: The search term to look for.
        max_results: Maximum number of results to return.
    """
    user = ctx.context
    return (
        f"Searching {user.username}'s data for '{query}'. "
        f"Found {max_results} results for user {user.user_id}."
    )


print("JSON schema for search_user_data:")
print(json.dumps(search_user_data.params_json_schema, indent=2))

## Cell 9 — Using Context for Business Logic: Subscription Tier Gating

Context is not just for data access — it is ideal for access control, feature flags, rate limiting, and any logic you want to keep completely invisible to the model. The model calls the tool, the tool inspects the context, and returns different results depending on the user's tier. The model is unaware the check is happening.

We run the same agent twice with two different context objects — one Free user and one Premium user — to demonstrate both code paths. The agent definition, instructions, and tool are identical in both runs.

In [ ]:
@function_tool
def generate_premium_report(
    ctx: RunContextWrapper[UserContext],
    topic: str,
) -> str:
    """Generates a detailed report on a given topic.

    Args:
        topic: The topic to generate a report on.
    """
    user = ctx.context
    if user.subscription_tier != "Premium":
        return (
            f"Sorry, {user.username}. "
            "Detailed reports are a Premium feature. "
            "Please upgrade your subscription to access this."
        )
    return (
        f"Premium report generated for {user.username} "
        f"on topic: {topic}. "
        "[Detailed analysis with 10 sections]"
    )


report_agent = Agent(
    name="Report Agent",
    instructions=(
        "You are a report generation assistant. "
        "Use the generate_premium_report tool when the user asks for a report."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[generate_premium_report],
)

# --- Free user ---
free_ctx = UserContext(
    user_id="u002",
    username="Rahul",
    subscription_tier="Free",
)
result_free = await Runner.run(
    report_agent,
    "Generate a report on market trends in AI.",
    context=free_ctx,
)
print("Free user result:", result_free.final_output)

print()

# --- Premium user ---
premium_ctx = UserContext(
    user_id="u001",
    username="Priya",
    subscription_tier="Premium",
)
result_premium = await Runner.run(
    report_agent,
    "Generate a report on market trends in AI.",
    context=premium_ctx,
)
print("Premium user result:", result_premium.final_output)

## Cell 10 — `ToolContext`: Accessing Tool-Specific Metadata

When you need more than just your app state — for example, to record exactly which tool was called, with which call ID, and with which raw arguments — switch the first parameter's type from `RunContextWrapper[T]` to `ToolContext[T]`.

`ToolContext` is a subclass of `RunContextWrapper`. It provides everything `RunContextWrapper` provides, plus the tool-call metadata fields listed in Cell 5. The SDK automatically creates a `ToolContext` from the `RunContextWrapper` for every function tool call.

Practical uses:
- **Audit logging**: record `tool_call_id`, `tool_name`, `tool_arguments` to a database
- **Idempotency**: use `tool_call_id` to detect and safely skip duplicate tool invocations
- **Debugging**: inspect raw arguments the model passed before your parsing logic runs

Remember: `ToolContext` is imported from `agents.tool_context`, not from the top-level `agents` package.

In [ ]:
@function_tool
def audited_action(
    ctx: ToolContext[UserContext],
    action: str,
) -> str:
    """Performs a user action and writes a full audit record.

    Args:
        action: A description of the action to perform.
    """
    user = ctx.context
    print(f"[AUDIT] tool_name      : {ctx.tool_name}")
    print(f"[AUDIT] tool_call_id   : {ctx.tool_call_id}")
    print(f"[AUDIT] tool_arguments : {ctx.tool_arguments}")
    print(f"[AUDIT] user           : {user.username} ({user.user_id})")
    return f"Action '{action}' completed for {user.username}."


audit_agent = Agent(
    name="Audit Agent",
    instructions=(
        "You are a helpful assistant. "
        "Use the audited_action tool to perform user requests."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[audited_action],
)

result = await Runner.run(
    audit_agent,
    "Export all my data.",
    context=premium_ctx,
)
print("Final output:", result.final_output)

## Cell 11 — Mutating Context Across Tool Calls

All tools in a single run share the **same** context instance. That means any mutation one tool makes to the context object is immediately visible to the next tool call in the same run. This makes the context object a natural accumulator for cross-call state — no external session store needed.

Here we use a `SessionContext` with two mutable fields:
- `tool_calls_made`: a list that every tool call appends to
- `total_items_fetched`: a running counter that every tool call increments

After the run we inspect both fields directly on the Python object to verify they were updated.

**Thread-safety reminder:** The context is not thread-safe. If you use `asyncio.gather()` to run multiple agents concurrently, each run must receive its own dedicated context instance. Sharing one instance across concurrent runs produces race conditions.

In [ ]:
@dataclass
class SessionContext:
    user_id: str
    tool_calls_made: list
    total_items_fetched: int


@function_tool
def fetch_items(
    ctx: RunContextWrapper[SessionContext],
    category: str,
    count: int,
) -> str:
    """Fetches items from a named category.

    Args:
        category: The category to fetch items from.
        count: The number of items to fetch.
    """
    session = ctx.context
    # Mutate context — visible to all subsequent tool calls in this run
    session.tool_calls_made.append(f"fetch_items:{category}")
    session.total_items_fetched += count
    return f"Fetched {count} items from '{category}'."


session_agent = Agent(
    name="Session Agent",
    instructions=(
        "You are a helpful assistant. "
        "Fetch items from multiple categories as requested by the user."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[fetch_items],
)

session = SessionContext(
    user_id="u001",
    tool_calls_made=[],
    total_items_fetched=0,
)

result = await Runner.run(
    session_agent,
    "Please fetch 3 items from electronics and 5 items from books.",
    context=session,
)

print("Agent output  :", result.final_output)
print("Tool calls    :", session.tool_calls_made)
print("Items fetched :", session.total_items_fetched)

## Cell 12 — Decision Guide: Which Context Type to Use?

Use this table to choose the right first parameter for any tool function you write:

| First parameter | Use when |
|-----------------|----------|
| `RunContextWrapper[T]` | You only need `ctx.context` (your app state) and optionally `ctx.usage`. Covers the majority of tool use cases. |
| `ToolContext[T]` | You also need `ctx.tool_name`, `ctx.tool_call_id`, or `ctx.tool_arguments` for auditing, logging, or idempotency checks. |
| No context parameter | The tool is purely computational and requires no runtime state from the run. |

### Where this pattern appears across the SDK

The `RunContextWrapper` pattern is not limited to function tools. You will encounter it in:

| SDK surface | Context type received |
|-------------|----------------------|
| `@function_tool` first parameter | `RunContextWrapper[T]` or `ToolContext[T]` |
| Dynamic instructions callback (Lecture 2.4) | `RunContextWrapper[T]` |
| Lifecycle hooks — `on_tool_start`, `on_tool_end` (Lecture 6.4) | `RunContextWrapper[T]` (castable to `ToolContext[T]`) |
| Input guardrails (Section 5) | `RunContextWrapper[T]` |

Learn the pattern once here and you will recognise it everywhere.